In [1]:
from z3 import *
from utils import * 
import sympy as sp
import numpy as np
from itertools import product
from fractions import Fraction
import random

In [2]:
# Number of Vertices
n = 3

# Generate DAGs
G1 = nx.DiGraph()
G2 = nx.DiGraph()
V = range(1,n+1)
G1.add_nodes_from(V)
G2.add_nodes_from(V)

# Edge Sets
E1 = [(1,2),(2,3)]
E2 = [(1,2),(1,3)]
B1 = [(2,3)]
B2 = [(2,3)]

# Add Edges
G1.add_edges_from(E1)
G2.add_edges_from(E2)

In [60]:
def create_z3_variables(E):
    return {
        (u, v): Real(f"x_{u}_{v}")
        for u, v in E
    }

E2 = [(1, 2), (1, 3)]

x = create_z3_variables(E2)
x

{(1, 2): x_1_2, (1, 3): x_1_3}

In [174]:
def create_random_B(E, n):
    L = sp.zeros(n, n)

    for u, v in E:
        # random rational number
        numerator = random.randint(-1000, 1000)
        denominator = random.randint(1, 1000)
        L[u-1, v-1] = sp.Rational(numerator, denominator)

    I = sp.eye(n)
    return (I - L).inv().T

B = create_random_B(E1, n)
B

Matrix([
[    1,        0, 0],
[21/34,        1, 0],
[-16/7, -544/147, 1]])

In [176]:
def create_equation(i, j, n, x_vars, B):

    expr = B[i-1,j-1]   # constant term

    for k in range(n):
        if (k+1, i) in x_vars and B[k,j-1]!=0:
            expr -= x_vars[(k+1, i)] * B[k,j-1]

    return expr

In [215]:
eq = create_equation(2, 1, n, x, B)
eq == 0

21/34 - x_1_2*1 == 0

In [184]:
def get_J(B2, n):
    B2 = set(B2)
    return [(i, j) for i in range(1, n+1)
                   for j in range(i+1, n+1)
                   if (i, j) not in B2]

def get_K(B1, n):
    symmetric_B1 = set(B1) | {(j, i) for i, j in B1}
    return [(i, j)
            for i in range(1, n+1)
            for j in range(1, n+1)
            if i == j or (i, j) in symmetric_B1]

def map_pairs(JK):
    return [((a, c), (b, d)) for ((a, b), (c, d)) in JK]

In [186]:
J = get_J(B2,n)
K = get_K(B1,n)
JK = map_pairs(list(product(J,K)))
JK[1]

((1, 2), (2, 2))

In [216]:
def create_variables(edge_list):
    return {(u, v): sp.Symbol(f"m_{u}_{v}") for u, v in edge_list}


def get_M_matrix(edge_list, n):
    vars = create_variables(edge_list)
    M = sp.zeros(n)

    for (u, v), var in vars.items():
        M[u-1, v-1] = var

    return M, list(vars.values())

def get_L_matrix(edge_list, n, low=-1.0, high=1.0):
    L = np.zeros((n, n))

    for u, v in edge_list:
        L[u-1, v-1] = np.random.uniform(low, high)

    return L

def get_J(B2, n):
    B2 = set(B2)
    return [(i, j) for i in range(1, n+1)
                   for j in range(i+1, n+1)
                   if (i, j) not in B2]

def get_K(B1, n):
    symmetric_B1 = set(B1) | {(j, i) for i, j in B1}
    return [(i, j)
            for i in range(1, n+1)
            for j in range(1, n+1)
            if i == j or (i, j) in symmetric_B1]

def map_pairs(JK):
    return [((a, c), (b, d)) for ((a, b), (c, d)) in JK]

def filter_pairs(JK, A):
    return [((a, b), (c, d)) 
            for ((a, b), (c, d)) in JK
            if A[a-1, b-1] != 0 and A[c-1, d-1] != 0]

def create_equations(JK, A):
    return [
        A[a-1, b-1] * A[c-1, d-1]
        for ((a, b), (c, d)) in JK
    ]

def get_both_linear_equations(JK, A):
    return [
        [A[a-1, b-1] , A[c-1, d-1]]
        for ((a, b), (c, d)) in JK
    ]

In [217]:
L = get_L_matrix(E1, n)
M, vars = get_M_matrix(E2, n)
I = np.eye(n)
A = (I-M).T @ np.linalg.inv(I - L).T
JK = map_pairs(list(product(J,K)))
JK_reduced = filter_pairs(JK, A)
JK_reduced

[((1, 1), (2, 1)), ((1, 1), (3, 1))]

In [227]:
vars

[m_1_2, m_1_3]

In [222]:
A

Matrix([
[                         1.0,                  0,   0],
[0.33601837991251 - 1.0*m_1_2,                1.0,   0],
[  -1.0*m_1_3 - 0.31640315162, -0.941624537629109, 1.0]])

In [229]:
constant_indices = [
    (i+1, j+1)
    for i in range(A.rows)
    for j in range(A.cols)
    if not (A[i, j].free_symbols & set(vars))
]
constant_indices

[(1, 1), (1, 2), (1, 3), (2, 2), (2, 3), (3, 2), (3, 3)]

In [230]:
(1,1) in constant_indices

True

In [245]:
s = Solver()

for (a,b),(c,d) in JK_reduced:

    if (a,b) in constant_indices:
        if (c,d) in constant_indices:
            print (False)
        else:
            s.add(create_equation(c, d, n, x, B)==0)
    elif (c,d) in constant_indices:
        s.add(create_equation(a,b,n,x,B)==0)
    else:    
        f = create_equation(a, b, n, x, B)
        g = create_equation(c, d, n, x, B)
        b = Bool(f"choose_{(a,b,c,d)}")
        s.add(If(b, f == 0, g == 0))

print(s.check())

sat
